In [1]:
from google.colab import drive
drive.mount('/content/drive')


MessageError: Error: credential propagation was unsuccessful

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat


In [ ]:
def load_mat_variable(mat_path, var_name=None):
    data = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    if var_name is None:
        keys = [k for k in data.keys() if not k.startswith('__')]
        if len(keys) == 1:
            return data[keys[0]]
        return {k: data[k] for k in keys}
    return data[var_name]

def parse_id_string(s):
    if pd.isna(s):
        return np.array([], dtype=np.uint64)
    if not isinstance(s, str):
        s = str(s)
    tokens = [tok for tok in s.replace(',', ' ').split() if tok]
    if not tokens:
        return np.array([], dtype=np.uint64)
    return np.array([int(float(tok)) for tok in tokens], dtype=np.uint64)

def safe_concat_uint64(arrays):
    arrays = [np.asarray(a, dtype=np.uint64).ravel() for a in arrays if len(a) > 0]
    if not arrays:
        return np.array([], dtype=np.uint64)
    return np.concatenate(arrays)

def to_1d_numeric_array(x, dtype=None):
  arr = np.asarray(x).squeeze()
  if arr.ndim == 0:
      arr = np.array([arr])
  if dtype is not None:
      arr = arr.astype(dtype)
  return arr

def pad_with_nan(arr, final_len):
    arr = np.asarray(arr, dtype=float).reshape(-1)
    if len(arr) < final_len:
        arr = np.concatenate([arr, np.full(final_len - len(arr), np.nan)])
    return arr

def matlab_struct_field_as_index(calls, field_name):
    vals = np.array([int(getattr(c, field_name)) for c in np.ravel(calls)])
    return np.where(vals == 1)[0] + 1


In [ ]:
try:
    BASE = next(Path('/content/drive/MyDrive').rglob('Fish1 Data_TEN_analysis'))
    print("Auto-detected:", BASE)
except StopIteration:
    BASE = Path(input("Paste path to SynapticInformation folder: ")) # /content/drive/MyDrive/Fish1 Data_TEN_analysis


outgoing_synapses_csv = BASE / 'outgoing_synapses.csv'
TEN_cats_csv = BASE / 'TEN_cats.csv'
synapse_sizes_csv = BASE / 'synapse_sizes.csv'

dmv_ids = BASE / 'DMVIDs.csv'

dmv_input_ids = BASE / 'DMV_InputsIDs.csv'
agglo_csv = BASE / 'agglomerated_segments_and_soma_ids.csv'




In [ ]:

data = pd.read_csv(outgoing_synapses_csv)
numRows = len(data)

pre_cell = data['query_neuron'].astype(np.uint64).tolist()
post_partners = [parse_id_string(s) for s in data['outgoing_with_soma']]

partners_with_soma = [
    {
        'presynaptic_id': np.uint64(pre),
        'postsynaptic_id': np.asarray(posts, dtype=np.uint64)
    }
    for pre, posts in zip(pre_cell, post_partners)
]

len(partners_with_soma), partners_with_soma[0]

TENdata = pd.read_csv(TEN_cats_csv)

TENIDs = TENdata.to_numpy().ravel()

# Rebuild TEN_categories from the CSV exactly as in MATLAB.
TEN_categories_rows = []
varNames = list(TENdata.columns)
for i in range(4):
    currVar = varNames[i]
    lst = pd.to_numeric(TENdata[currVar], errors='coerce').dropna().to_numpy()
    combined = np.column_stack([
        lst.reshape(-1, 1),
        np.full((len(lst), 1), i + 1)
    ])
    TEN_categories_rows.append(combined)

TEN_categories = np.vstack(TEN_categories_rows).astype(np.uint64)


BadIDs = np.array([16122838713, 11592362939, 15976220988, 16098552493, 16098551160], dtype=np.uint64)
TENIDs = TENIDs[~np.isin(TENIDs, BadIDs)]
TENIDs = TENIDs[TENIDs != 0]
TEN_categories = TEN_categories[~np.isin(TEN_categories[:, 0], BadIDs)]

Type = {}
for i in range(1, 5):
    Type[i] = TEN_categories[TEN_categories[:, 1] == i, 0].astype(np.uint64)

for k in range(1, 5):
    print(k, len(Type[k]))


In [ ]:
specific_neurons = TENIDs.copy()

inh_flag = 0
both_flag = 0
exc_flag = 0
ipsi_inh_mons = 0
remove_single_mons = 0
layout_type = 'force'

specific_set = set(specific_neurons.tolist())
filtered_partners = [p for p in partners_with_soma if p['presynaptic_id'] in specific_set]
presynaptic_ids = np.array([p['presynaptic_id'] for p in filtered_partners], dtype=np.uint64)

# Equivalent of the MATLAB loop for Presyn_category
TEN_cat_map = {int(row[0]): int(row[1]) for row in TEN_categories}
Presyn_category = np.array([TEN_cat_map.get(int(pid), np.nan) for pid in presynaptic_ids], dtype=float)

len(filtered_partners), presynaptic_ids[:5], Presyn_category[:5]

In [ ]:
VagalList = [9981263094, 11286751116, 11307407543, 11347760939, 11347888097, 11491189063, 11511524388, 11552065799, 11552128855, 11552323117, 12715018111, 12816692072, 12816693902, 12857492095, 13000857312, 13061863959, 13062249324, 13102153639, 13103233032, 13123256654, 13143399261, 13143463889, 13143591174, 13163928771, 14326489686, 14347016159, 14388195599, 14571850576, 14612330497, 14652499833, 14653065838, 14653322070, 14674042236, 14755323257, 14775657653, 15816144378, 15816144800, 15876901565, 15897419245, 15918082043, 16122838713, 16183013914, 16203481119, 16223244922, 16244021793, 16244023987, 16244083137, 16244153555, 16244217341, 16264553829, 16366347970, 17263850313, 17284255460, 17304531160, 17325251226, 17325559934, 17345836956, 17346084295, 17366362600, 17386889207, 20691186392]
SpinalList = [12836468517, 14407389022, 16060234912, 17365352671, 17406013023, 17487804938, 17488878969, 17509151175, 17548693066, 17569023577, 17569291344, 17693076200, 17710420602, 17713603533, 18853724298, 18853725983, 18916389471, 18976887013, 18977392277, 19018318081, 19039606516, 19161358803, 20507462749, 20527477307, 20548388338, 20568531285, 20588869488, 20649433245, 20649822944, 22098533754, 22160367563]
IOList = [6552943218, 6573210745, 6573271951, 6593673947, 6593674161, 6593674164, 6654868684, 6675270958, 6675271281, 7144517890, 7144518051, 7144518121, 7164852175, 8143941688, 8164348591, 15671187907, 17180731805, 17834597166, 19364409543, 20363965444, 20670468135, 20711329385, 20711959185, 20833726750, 20833981112, 20833981513, 20854126293, 20854252548, 20873949136, 20874398957, 20894112831, 20894117727, 21894423017, 21915004454, 21935601556, 21996731214, 21996796675, 22037660749, 22098668033, 22098668287, 22098668390, 22098977609, 22119058724, 22139340169, 22139341313, 22159544972, 22160063467, 22160065196, 22180399541, 22180463300, 22200532921, 22200600059, 22200727299, 22200796331, 22384131360, 22404341603, 22404341773, 23506211092, 23587938455, 23588001187, 23588001595, 23588064880, 23608341643, 23628542059, 23628679530, 23628801958, 23628927042, 23648883770, 23648885173, 23669281986, 23669408941, 23669410067, 23669658970, 23710078200, 23710197184, 23710197231, 23710388607, 23710455694, 23710457525, 23730341223, 23730601825, 23730605502, 23730659731, 23730726993, 23730859567, 23751071952, 23751122764, 23751134262, 23751134336, 23751197910, 23771344683, 23771407536, 23771409817, 23771472480, 23791870030, 23791872091, 25158545977, 25199409613]
LocalList =[11426688959, 11447156371, 12936995494, 12957330530, 14202769096, 14385463441, 14405107377, 14405862988, 14426135054, 14426136271, 14426196819, 14426259771, 14466804770, 14466931953, 14467063054, 14528191771, 14528259257, 14548465049, 14548592366, 14548594846, 14589072504, 14609727382, 14672902891, 14691453029, 15833809067, 15874542427, 15894876268, 15894941806, 15935554036, 15956015312, 15956137365, 15956137491, 15956141044, 15956203651, 15976221870, 15976600154, 15976601005, 15996814107, 15996940756, 15997000133, 15997065252, 16017080955, 16017146250, 16017207892, 16017207990, 16017271988, 16017273514, 16017401519, 16017464374, 16037418777, 16057943303, 16057945320, 16057946302, 16078406239, 16098614875, 16098870608, 16098936465, 16119272994, 16119331477, 16119333740, 16139671617, 16139731297, 16139794985, 16159939992, 16160195797, 16180470840, 16180655851, 16180659500, 16202529937, 16223113067, 16223175142, 16241854711, 16285014494, 17282155591, 17342842956, 17343158730, 17383644734, 17384083524, 17404357709, 17404614463, 17424761256, 17424946103, 17445096455, 17445415981, 17465429830, 17465431160, 17465431635, 17486024949, 17506294659, 17526625653, 17526688000, 17526824029, 17547088270, 17547218002, 17547342099, 17547343476, 17548693066, 17569291344, 17608475305, 17628747864, 17649284262, 17669294073, 17710420602, 17710476726, 17812406079, 17832804173, 18730561569, 18771172377, 18811776669, 18811966709, 18812098125, 18873037398, 18893439908, 18893690830, 18914089919, 18934488630, 18995763992, 19016096755, 19016102088, 19036371049, 19056702458, 19076969492, 19077162275, 19097561301, 19117830529, 19118029775, 19158830520, 19179033530, 19199178119, 19199369432, 19219902101, 19219956515, 19240239663, 19260945529, 19281283204, 19362498877, 19383019627, 19403610763, 20321576191, 20341784258, 20361802278, 20361935509, 20362118046, 20484403492, 20525585224, 20668170445, 20729050834, 20769410744, 20851899164, 20872293560, 21891876995, 21891940030, 21892125388, 21912272796, 21994379590, 22014147221, 22014531846, 22034994100, 22259130783, 22279335362, 22279336106, 22300305924, 22422322932, 22422386491, 22422387312, 22442973402, 23401934181, 23422202562, 23442667857, 23462874414, 23463068912, 23524321427, 23585399754, 23809472337, 23850138916, 24931307456, 24952143720, 24952204737, 24972038420, 24992363355, 25012964613, 25033486377, 25053761676, 25095006815, 25278533539, 25380341669, 25420944765, 26461174526, 26502105722, 26502164683, 27012458986, 28093364838]

In [ ]:
SynapseTable = pd.read_csv(synapse_sizes_csv)
import matplotlib.pyplot as plt
colors = [
    (0, 1, 0),  # Type 1
    (0, 1, 1),  # Type 2
    (1, 0, 1),  # Type 3
    (1, 0, 0)   # Type 4
]
for col in ['pre_neuron_id', 'post_id', 'bbx', 'bby', 'bbz']:
    SynapseTable[col] = pd.to_numeric(SynapseTable[col], errors='coerce')

SynapseTable = SynapseTable.dropna(subset=['pre_neuron_id', 'post_id', 'bbx', 'bby', 'bbz']).copy()
SynapseTable['pre_neuron_id'] = SynapseTable['pre_neuron_id'].astype(np.uint64)
SynapseTable['post_id'] = SynapseTable['post_id'].astype(np.uint64)
SynapseTable['volume'] = ((8 * SynapseTable['bbx']) * (8 * SynapseTable['bby']) * (30 * SynapseTable['bbz'])).astype(np.uint64)

vol_thresh_list = [0,5000000, 10000000, 20000000, 30000000,40000000,50000000]
percent_in_IO = np.zeros((5,len(vol_thresh_list)), dtype=float)
percent_in_Local = np.zeros((5,len(vol_thresh_list)), dtype=float)
percent_in_Spinal = np.zeros((5,len(vol_thresh_list)), dtype=float)
percent_in_Vagal = np.zeros((5,len(vol_thresh_list)), dtype=float)
count = 0
for vol_thresh in vol_thresh_list:
  # MATLAB uses max(volume_for_pair) > threshold, which is equivalent to:
  # pair is present if ANY synapse between the pair exceeds threshold.
    good_pairs = (
        SynapseTable.groupby(['pre_neuron_id', 'post_id'], as_index=False)['volume']
        .max()
    )
    good_pairs = good_pairs[good_pairs['volume'] > vol_thresh][['pre_neuron_id', 'post_id']].copy()
    good_pairs['edge_value'] = 1.0
    unique_postsynaptic_ids = np.unique(safe_concat_uint64([p['postsynaptic_id'] for p in filtered_partners]))
    combined_ids = np.concatenate([presynaptic_ids, unique_postsynaptic_ids]).astype(np.uint64)

  # MATLAB containers.Map keeps one entry per key. If combined_ids contains duplicates,
  # later occurrences overwrite earlier ones. Build the dict the same way.
    id_index_map = {int(cell_id): idx for idx, cell_id in enumerate(combined_ids)}

  # Explode requested edges from filtered_partners
    edge_rows = []
    for p in filtered_partners:
      pre = int(p['presynaptic_id'])
      for post in p['postsynaptic_id']:
          edge_rows.append((pre, int(post)))

    requested_edges = pd.DataFrame(edge_rows, columns=['pre_neuron_id', 'post_id'])

  # Keep only edges whose pair has max volume > threshold
    edge_counts = requested_edges.merge(good_pairs, on=['pre_neuron_id', 'post_id'], how='inner')
    edge_counts['pre_idx'] = edge_counts['pre_neuron_id'].map(id_index_map)
    edge_counts['post_idx'] = edge_counts['post_id'].map(id_index_map)
    connectivity_matrix = np.zeros((len(combined_ids), len(combined_ids)), dtype=float)
    for row in edge_counts[['pre_idx', 'post_idx']].itertuples(index=False):
        connectivity_matrix[row.pre_idx, row.post_idx] += 1
    connectivity_matrix.shape
    NoTENcons = connectivity_matrix[:111, :].sum(axis=0)
    connectivity_matrix[:, NoTENcons < 1] = 0
    PostByCell = {}
    TypePost = {}
    for i in range(1, 5):
        TypePost[i] = np.array([], dtype=np.uint64)
        for j, currTEN in enumerate(Type[i], start=1):
            idxcurr = np.where(combined_ids == currTEN)[0]
            if len(idxcurr) == 0:
                continue
            idxcurr = idxcurr[0]
            post_loc = np.where(connectivity_matrix[idxcurr, :] != 0)[0]
            if len(post_loc) > 0:
                posts = combined_ids[post_loc].astype(np.uint64)
                PostByCell[(i, j)] = posts
                TypePost[i] = np.unique(np.concatenate([TypePost[i], posts]))
        percent = np.isin(TypePost[i], list(IOList)).sum() / len(TypePost[i])
        percent_in_IO[i,count] = percent
        percent = np.isin(TypePost[i], list(LocalList)).sum() / len(TypePost[i])
        percent_in_Local[i,count] = percent
        percent = np.isin(TypePost[i], list(SpinalList)).sum() / len(TypePost[i])
        percent_in_Spinal[i,count] = percent
        percent = np.isin(TypePost[i], list(VagalList)).sum() / len(TypePost[i])
        percent_in_Vagal[i,count] = percent
    Overlap = np.full((4, 4), np.nan, dtype=float)
    for i in range(1, 5):
        for j in range(1, 5):
            denom = len(TypePost[i])
            if denom > 0:
                Overlap[i-1, j-1] = np.isin(TypePost[i], TypePost[j]).sum() / denom
    plt.figure(figsize=(6,6))  # square figure
    fig, ax = plt.subplots()
    im = ax.imshow(Overlap, vmin=0, vmax=1, origin='upper', aspect='equal')
    for i in range(Overlap.shape[0]):
        for j in range(Overlap.shape[1]):
            ax.text(j, i, f'{Overlap[i, j]:.003f}',
                    ha='center', va='center', color='red')

    plt.colorbar(im)

    #plt.colorbar(label='Fraction of Overlap')
    plt.xticks([0,1,2,3], ['Class I','Clas II','Class III','Class IV'], rotation=45)
    plt.yticks([0,1,2,3], ['Class I','Clas II','Class III','Class IV'])
    plt.savefig('ClassOverlap'+ str(count)+ '.pdf', format='pdf')
    plt.close()
    count+=1

In [ ]:

for t in range(1,5):
    plt.plot(percent_in_IO[t,:], color=colors[t-1], marker='o', label=f'Type {t}')
labels = [x / 1000000 for x in vol_thresh_list]
ML = 18
LL = 12
plt.xticks(range(len(labels)), labels, fontsize=LL)
plt.yticks(fontsize=LL)
plt.xlabel('Synaptic Size Threshold', fontsize=ML)
plt.ylabel('Fraction of Inferior Olive Targets', fontsize=ML)
plt.legend(['Class I','Class II','Class III','Class IV'])
plt.ylim(0,1)
plt.savefig('IOTargets.pdf', format='pdf')
plt.show()

for t in range(1,5):
    plt.plot(percent_in_Local[t,:], color=colors[t-1], marker='o', label=f'Type {t}')
plt.xticks(range(len(labels)), labels, fontsize=LL)
plt.yticks(fontsize=LL)
plt.xlabel('Synaptic Size Threshold', fontsize=ML)
plt.ylabel('Fraction of Midbrain Targets', fontsize=ML)
plt.legend(['Class I','Clas II','Class III','Class IV'])
plt.ylim(0,1)
plt.savefig('MidbrainTargets.pdf', format='pdf')
plt.show()

for t in range(1,5):
    plt.plot(percent_in_Spinal[t,:], color=colors[t-1], marker='o', label=f'Type {t}')
plt.xticks(range(len(labels)), labels, fontsize=LL)
plt.yticks(fontsize=LL)
plt.xlabel('Synaptic Size Threshold', fontsize=ML)
plt.ylabel('Fraction of Spinal Projection Targets', fontsize=ML)
plt.legend(['Class I','Clas II','Class III','Class IV'])
plt.ylim(0,1)
plt.savefig('SpinalTargets.pdf', format='pdf')
plt.show()

for t in range(1,5):
    plt.plot(percent_in_Vagal[t,:], color=colors[t-1], marker='o', label=f'Type {t}')
plt.xticks(range(len(labels)), labels, fontsize=LL)
plt.yticks(fontsize=LL)
plt.xlabel('Synaptic Size Threshold', fontsize=ML)
plt.ylabel('Fraction of Vagal Region Targets', fontsize=ML)
plt.legend(['Class I','Clas II','Class III','Class IV'])
plt.ylim(0,1)
plt.savefig('VagalTargets.pdf', format='pdf')
plt.show()

In [ ]:
max_len = max(len(TypePost[t]) for t in range(1, 5))

# build columns
cols = {}
for t in range(1, 5):
    arr = np.array(TypePost[t], dtype=float)
    if len(arr) < max_len:
        arr = np.concatenate([arr, np.full(max_len - len(arr), np.nan)])
    cols[f'Type{t}'] = arr

# create dataframe
TypePost_table = pd.DataFrame(cols)

# save
out_path = '/content/TypePost_table.csv'
TypePost_table.to_csv(out_path, index=False)

# download (Colab)
from google.colab import files
#files.download(out_path)

In [ ]:
Overlap = np.full((4, 4), np.nan, dtype=float)
for i in range(1, 5):
    for j in range(1, 5):
        denom = len(TypePost[i])
        if denom > 0:
            Overlap[i-1, j-1] = np.isin(TypePost[i], TypePost[j]).sum() / denom
plt.figure(figsize=(6,6))  # square figure
fig, ax = plt.subplots()
im = ax.imshow(Overlap, vmin=0, vmax=1, origin='upper', aspect='equal')
for i in range(Overlap.shape[0]):
    for j in range(Overlap.shape[1]):
        ax.text(j, i, f'{Overlap[i, j]:.003f}',
                ha='center', va='center', color='red')

plt.colorbar(im)

#plt.colorbar(label='Fraction of Overlap')
plt.xticks([0,1,2,3], ['Class I','Clas II','Class III','Class IV'], rotation=45)
plt.yticks([0,1,2,3], ['Class I','Clas II','Class III','Class IV'])
plt.savefig('ClassOverlap.pdf', format='pdf')
plt.show()





In [ ]:
# Generate PAG -> DMV data for a single synapse size threshold
volume_threshold = 30000000

DMV_ids = np.loadtxt(dmv_ids, delimiter=',', dtype=np.uint64)
DMV_input_ids = np.loadtxt(dmv_input_ids, delimiter=',', dtype=np.uint64)
TypeIV_outs = TypePost[4]

BadIDs = np.array([
    16122838713, 11592362939, 15976220988, 16098552493, 16098551160
], dtype=np.uint64)

#TEN_categories = TEN_categories[~np.isin(TEN_categories[:, 0].astype(np.uint64), BadIDs)]
#Type = {i: TEN_categories[TEN_categories[:, 1] == i, 0].astype(np.uint64) for i in range(1, 5)}

specific_neurons = np.concatenate([
    Type[4].ravel(),
    TypeIV_outs.ravel(),
    DMV_input_ids.ravel(),
    DMV_ids.ravel(),
]).astype(np.uint64)

combined_ids = specific_neurons.copy()  # mirrors the final MATLAB overwrite
combined_id_set = set(map(int, combined_ids))

inh_flag = 0
both_flag = 0
exc_flag = 0
ipsi_inh_mons = 0
remove_single_mons = 0
layout_type = 'force'

partners_df = pd.read_csv(outgoing_synapses_csv)
partners_df = partners_df.rename(columns={'query_neuron': 'pre_id', 'outgoing_with_soma': 'post_list'})
partners_df['pre_id'] = partners_df['pre_id'].astype('uint64')
partners_df['post_list'] = partners_df['post_list'].map(parse_id_string)

# Keep only the presynaptic neurons of interest.
partners_df = partners_df[partners_df['pre_id'].isin(specific_neurons)].copy()

# Explode into one row per edge.
edges = partners_df[['pre_id', 'post_list']].explode('post_list', ignore_index=True)
edges = edges.rename(columns={'post_list': 'post_id'})
edges = edges.dropna(subset=['post_id']).copy()
edges['post_id'] = edges['post_id'].astype('uint64')

edges.head(), len(edges)
SynapseTable = pd.read_csv(synapse_sizes_csv)
SynapseTable = SynapseTable.dropna(subset=['pre_neuron_id'])
SynapseTable = SynapseTable.dropna(subset=['post_id'])
SynapseTable['pre_id'] = SynapseTable['pre_neuron_id'].astype('uint64')
SynapseTable['post_id'] = SynapseTable['post_id'].astype('uint64')
SynapseTable['volume'] = (8 * SynapseTable['bbx']) * (8 * SynapseTable['bby']) * (30 * SynapseTable['bbz'])

max_volume = (
    SynapseTable[['pre_id', 'post_id', 'volume']]
    .groupby(['pre_id', 'post_id'], as_index=False)['volume']
    .max()
)

max_volume.head(), len(max_volume)
edges = edges.merge(max_volume, on=['pre_id', 'post_id'], how='left')
edges = edges[edges['volume'].fillna(-1) > volume_threshold].copy()

edges = edges[edges['post_id'].isin(combined_ids)].copy()

edges.head(), len(edges)

combined_ids_unique = pd.Index(pd.unique(combined_ids))
id_to_index = pd.Series(
    np.arange(len(combined_ids_unique), dtype=np.int64),
    index=combined_ids_unique
)
edge_counts = (
    edges.groupby(['pre_id', 'post_id'], as_index=False)
    .size()
    .rename(columns={'size': 'weight'})
)

#id_to_index = pd.Series(np.arange(len(combined_ids)), index=combined_ids)
#edge_counts['pre_idx'] = id_to_index.loc[edge_counts['pre_id']].to_numpy()
#edge_counts['post_idx'] = id_to_index.loc[edge_counts['post_id']].to_numpy()

id_to_index = {int(cell_id): i for i, cell_id in enumerate(pd.unique(combined_ids))}
edge_counts['pre_idx'] = edge_counts['pre_id'].map(id_to_index)
edge_counts['post_idx'] = edge_counts['post_id'].map(id_to_index)


connectivity_matrix = np.zeros((len(combined_ids), len(combined_ids)), dtype=np.int32)
connectivity_matrix[edge_counts['pre_idx'], edge_counts['post_idx']] = edge_counts['weight'].to_numpy()

connectivity_matrix.shape, connectivity_matrix.sum()
CM2 = connectivity_matrix.copy().astype(np.int64)
for i in range(1, connectivity_matrix.shape[0] - 1):
    for j in range(1, connectivity_matrix.shape[1] - 1):
        CM2[i, j] = CM2[i, j] + connectivity_matrix[i-1:i+2, j-1:j+2].sum()

sizes = [
    len(Type[4].ravel()),
    len(TypeIV_outs.ravel()),
    len(DMV_input_ids.ravel()),
    len(DMV_ids.ravel())
]

boundaries = np.cumsum(sizes)

boundary_lines = boundaries[:-1]
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))  # square figure
plt.imshow(CM2, vmin=0, vmax=5, cmap='gray', origin='upper', aspect='equal')
plt.colorbar(label='Connection strength')

for b in boundary_lines:
    plt.axhline(b - 0.5, linestyle='--')
    plt.axvline(b - 0.5, linestyle='--')

plt.title('Connectivity Matrix')
plt.xlabel('Post')
plt.ylabel('Pre')

plt.gca().set_aspect('equal', adjustable='box')  # enforce square axes

labels = ['Type IV', 'Type IV outputs', 'DMV inputs', 'DMV']
centers = np.cumsum(sizes) - np.array(sizes)/2

plt.xticks(centers, labels, rotation=45)
plt.yticks(centers, labels)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# block boundaries
# -----------------------------
n_type4 = len(Type[4].ravel())
n_type4outs = len(TypeIV_outs.ravel())
n_dmv_inputs = len(DMV_input_ids.ravel())
n_dmv = len(DMV_ids.ravel())

type4_start, type4_end = 0, n_type4
type4outs_start, type4outs_end = type4_end, type4_end + n_type4outs
dmv_inputs_start, dmv_inputs_end = type4outs_end, type4outs_end + n_dmv_inputs
dmv_start, dmv_end = dmv_inputs_end, dmv_inputs_end + n_dmv

# -----------------------------
# helper: inputs received by each DMV neuron from a source block
# -----------------------------
def inputs_to_dmv_from_source(mat, source_start, source_end, dmv_start, dmv_end):
    sub = mat[source_start:source_end, dmv_start:dmv_end]
    return sub.sum(axis=0)   # one value per DMV neuron

# -----------------------------
# helper: bootstrap median CI
# -----------------------------
def bootstrap_median_ci(x, n_boot=10000, ci=95, seed=0):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    samples = rng.choice(x, size=(n_boot, len(x)), replace=True)
    boot_medians = np.median(samples, axis=1)

    alpha = 100 - ci
    lower = np.percentile(boot_medians, alpha / 2)
    upper = np.percentile(boot_medians, 100 - alpha / 2)

    return np.median(x), lower, upper

# -----------------------------
# per-DMV-neuron input counts from each source group
# -----------------------------
vals_type4_to_dmv = inputs_to_dmv_from_source(
    connectivity_matrix, type4_start, type4_end, dmv_start, dmv_end
)

vals_type4outs_to_dmv = inputs_to_dmv_from_source(
    connectivity_matrix, type4outs_start, type4outs_end, dmv_start, dmv_end
)

vals_dmvinputs_to_dmv = inputs_to_dmv_from_source(
    connectivity_matrix, dmv_inputs_start, dmv_inputs_end, dmv_start, dmv_end
)

vals_dmv_to_dmv = inputs_to_dmv_from_source(
    connectivity_matrix, dmv_start, dmv_end, dmv_start, dmv_end
)

# -----------------------------
# filter DMV neurons:
# keep only those receiving at least one input from any category
# -----------------------------
total_inputs_to_dmv = (
    vals_type4_to_dmv +
    vals_type4outs_to_dmv +
    vals_dmvinputs_to_dmv +
    vals_dmv_to_dmv
)

keep = total_inputs_to_dmv > 0

vals_type4_to_dmv = vals_type4_to_dmv[keep]
vals_type4outs_to_dmv = vals_type4outs_to_dmv[keep]
vals_dmvinputs_to_dmv = vals_dmvinputs_to_dmv[keep]
vals_dmv_to_dmv = vals_dmv_to_dmv[keep]

groups = [
    vals_type4_to_dmv,
    vals_type4outs_to_dmv,
    vals_dmvinputs_to_dmv,
    vals_dmv_to_dmv
]

labels = [
    'Type IV → DMV',
    'TypeIV outs → DMV',
    'DMV_inputs → DMV',
    'DMV → DMV'
]

# -----------------------------
# bootstrap medians and 95% CI
# -----------------------------
medians = []
lowers = []
uppers = []

for k, g in enumerate(groups):
    m, lo, hi = bootstrap_median_ci(g, n_boot=10000, ci=95, seed=100 + k)
    medians.append(m)
    lowers.append(lo)
    uppers.append(hi)

medians = np.array(medians)
lowers = np.array(lowers)
uppers = np.array(uppers)

yerr = np.vstack([medians - lowers, uppers - medians])

# -----------------------------
# plot
# -----------------------------
plt.figure(figsize=(6, 5))
plt.bar(
    range(len(medians)),
    medians,
    yerr=yerr,
    color=(0.5, 0.5, 0.5),
    capsize=0
)

rng = np.random.default_rng(0)
for i, g in enumerate(groups):
    x = rng.normal(loc=i, scale=0.06, size=len(g))
    plt.scatter(x, g, color='black', s=20, alpha=0.6, zorder=3)

plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
plt.ylabel('Inputs per DMV neuron', fontsize=12)
plt.title('DMV neurons with at least one input from any category', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# --- rebuild from scratch, with fresh variable names ---
syn = pd.read_csv(synapse_sizes_csv).copy()
syn = syn.dropna(subset=['pre_neuron_id', 'post_id']).copy()

syn['pre_id'] = syn['pre_neuron_id'].astype(np.uint64)
syn['post_id'] = syn['post_id'].astype(np.uint64)
syn['syn_volume'] = (8 * syn['bbx']) * (8 * syn['bby']) * (30 * syn['bbz'])

pair_max_volume = (
    syn[['pre_id', 'post_id', 'syn_volume']]
    .groupby(['pre_id', 'post_id'], as_index=False)['syn_volume']
    .max()
    .rename(columns={'syn_volume': 'pair_max_volume'})
)

# sanity checks
print("pair_max_volume columns:", pair_max_volume.columns.tolist())
print("edges columns before merge:", edges.columns.tolist())

# remove any stale columns from edges before merging
edges_base = edges.drop(columns=['pair_max_volume', 'syn_volume', 'volume', 'volume_x', 'volume_y', 'max_volume'],
                        errors='ignore').copy()

edges_with_vol = edges_base.merge(
    pair_max_volume,
    on=['pre_id', 'post_id'],
    how='left',
    validate='many_to_one'
)

print("edges_with_vol columns after merge:", edges_with_vol.columns.tolist())

# now filter
volume_threshold = 28800000
edges_thr = edges_with_vol.loc[
    edges_with_vol['pair_max_volume'].fillna(-1) > volume_threshold
].copy()

print(edges_thr.head())
print("n edges after threshold:", len(edges_thr))
print(edges_with_vol.columns.tolist())
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# inputs assumed to already exist from your notebook:
# edges
# combined_ids
# Type
# TypeIV_outs
# DMV_input_ids
# DMV_ids
# synapse_sizes_csv
# --------------------------------------------------

# thresholds to test
volume_thresholds =  [0,5000000, 10000000, 20000000, 30000000,40000000,50000000]

# colors for the 4 source categories:
# Type IV, Type IV outputs, DMV inputs, DMV
colors = [
    (0, 0, 0),
    (1, 0, 0),
    (0, 1, 1),
    (0, 1, 0)
]

labels = [
    'Type IV → DMV',
    'Type IV outputs → DMV',
    'DMV inputs → DMV',
    'DMV → DMV'
]

# --------------------------------------------------
# load and preprocess synapse sizes once
# --------------------------------------------------
SynapseTable = pd.read_csv(synapse_sizes_csv).copy()
SynapseTable = SynapseTable.dropna(subset=['pre_neuron_id', 'post_id']).copy()
SynapseTable['pre_id'] = SynapseTable['pre_neuron_id'].astype(np.uint64)
SynapseTable['post_id'] = SynapseTable['post_id'].astype(np.uint64)
SynapseTable['syn_volume'] = (
    (8 * SynapseTable['bbx']) *
    (8 * SynapseTable['bby']) *
    (30 * SynapseTable['bbz'])
)

pair_max_volume = (
    SynapseTable[['pre_id', 'post_id', 'syn_volume']]
    .groupby(['pre_id', 'post_id'], as_index=False)['syn_volume']
    .max()
    .rename(columns={'syn_volume': 'pair_max_volume'})
)

# --------------------------------------------------
# block boundaries
# --------------------------------------------------
n_type4 = len(Type[4].ravel())
n_type4outs = len(TypeIV_outs.ravel())
n_dmv_inputs = len(DMV_input_ids.ravel())
n_dmv = len(DMV_ids.ravel())

type4_start, type4_end = 0, n_type4
type4outs_start, type4outs_end = type4_end, type4_end + n_type4outs
dmv_inputs_start, dmv_inputs_end = type4outs_end, type4outs_end + n_dmv_inputs
dmv_start, dmv_end = dmv_inputs_end, dmv_inputs_end + n_dmv

# keep only IDs that are actually in the matrix
combined_ids_unique = pd.Index(pd.unique(combined_ids))
id_to_index = {int(cell_id): i for i, cell_id in enumerate(combined_ids_unique)}

# --------------------------------------------------
# helpers
# --------------------------------------------------
def inputs_to_dmv_from_source(mat, source_start, source_end, dmv_start, dmv_end):
    block = mat[source_start:source_end, dmv_start:dmv_end]
    return block.sum(axis=0)  # one value per DMV neuron

def bootstrap_median_ci(x, n_boot=10000, ci=95, seed=0):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    samples = rng.choice(x, size=(n_boot, len(x)), replace=True)
    boot_medians = np.mean(samples, axis=1)

    alpha = 100 - ci
    lower = np.percentile(boot_medians, alpha / 2)
    upper = np.percentile(boot_medians, 100 - alpha / 2)

    return np.mean(x), lower, upper

# --------------------------------------------------
# storage for results
# --------------------------------------------------
n_groups = 4
medians_by_group = [[] for _ in range(n_groups)]
lowers_by_group = [[] for _ in range(n_groups)]
uppers_by_group = [[] for _ in range(n_groups)]

# --------------------------------------------------
# loop over thresholds
# --------------------------------------------------
for t_idx, volume_threshold in enumerate(volume_thresholds):
    # start from a clean copy of edges each time
    edges_base = edges.drop(
        columns=['pair_max_volume', 'syn_volume', 'volume', 'volume_x', 'volume_y', 'max_volume'],
        errors='ignore'
    ).copy()

    # add max volume per (pre, post) pair
    edges_with_vol = edges_base.merge(
        pair_max_volume,
        on=['pre_id', 'post_id'],
        how='left',
        validate='many_to_one'
    )

    # threshold by pair max volume
    edges_thr = edges_with_vol[
        edges_with_vol['pair_max_volume'].fillna(-1) > volume_threshold
    ].copy()

    # keep only IDs that exist in the matrix
    edges_thr = edges_thr[edges_thr['post_id'].isin(combined_ids_unique)].copy()
    edges_thr = edges_thr[edges_thr['pre_id'].isin(combined_ids_unique)].copy()

    # count multiplicity of surviving edges
    edge_counts = (
        edges_thr.groupby(['pre_id', 'post_id'], as_index=False)
        .size()
        .rename(columns={'size': 'weight'})
    )

    edge_counts['pre_idx'] = edge_counts['pre_id'].map(id_to_index)
    edge_counts['post_idx'] = edge_counts['post_id'].map(id_to_index)

    # drop any rows that failed to map
    edge_counts = edge_counts.dropna(subset=['pre_idx', 'post_idx']).copy()
    edge_counts['pre_idx'] = edge_counts['pre_idx'].astype(int)
    edge_counts['post_idx'] = edge_counts['post_idx'].astype(int)

    # build connectivity matrix
    connectivity_matrix = np.zeros((len(combined_ids_unique), len(combined_ids_unique)), dtype=np.int32)

    # accumulate weights safely
    np.add.at(
        connectivity_matrix,
        (edge_counts['pre_idx'].to_numpy(), edge_counts['post_idx'].to_numpy()),
        edge_counts['weight'].to_numpy()
    )

    # per-DMV-neuron input counts from each source group
    vals_type4_to_dmv = inputs_to_dmv_from_source(
        connectivity_matrix, type4_start, type4_end, dmv_start, dmv_end
    )
    vals_type4outs_to_dmv = inputs_to_dmv_from_source(
        connectivity_matrix, type4outs_start, type4outs_end, dmv_start, dmv_end
    )
    vals_dmvinputs_to_dmv = inputs_to_dmv_from_source(
        connectivity_matrix, dmv_inputs_start, dmv_inputs_end, dmv_start, dmv_end
    )
    vals_dmv_to_dmv = inputs_to_dmv_from_source(
        connectivity_matrix, dmv_start, dmv_end, dmv_start, dmv_end
    )

    # keep only DMV neurons receiving at least one input from any category
    total_inputs_to_dmv = (
        vals_type4_to_dmv +
        vals_type4outs_to_dmv +
        vals_dmvinputs_to_dmv +
        vals_dmv_to_dmv
    )
    keep = total_inputs_to_dmv > 0

    groups = [
        vals_type4_to_dmv[keep],
        vals_type4outs_to_dmv[keep],
        vals_dmvinputs_to_dmv[keep],
        vals_dmv_to_dmv[keep]
    ]

    # bootstrap median + 95% CI for each group
    for g_idx, g in enumerate(groups):
        med, lo, hi = bootstrap_median_ci(
            g,
            n_boot=10000,
            ci=95,
            seed=1000 * t_idx + g_idx
        )
        medians_by_group[g_idx].append(med)
        lowers_by_group[g_idx].append(lo)
        uppers_by_group[g_idx].append(hi)

# convert to arrays
medians_by_group = [np.asarray(x, dtype=float) for x in medians_by_group]
lowers_by_group = [np.asarray(x, dtype=float) for x in lowers_by_group]
uppers_by_group = [np.asarray(x, dtype=float) for x in uppers_by_group]

# --------------------------------------------------
# plot: median inputs vs threshold
# --------------------------------------------------
plt.figure(figsize=(7, 5))

for i in range(n_groups):
    y = medians_by_group[i]
    yerr = np.vstack([y - lowers_by_group[i], uppers_by_group[i] - y])

    plt.errorbar(
        volume_thresholds,
        y,
        yerr=yerr,
        color=colors[i],
        marker='o',
        linewidth=2,
        elinewidth=1.2,
        capsize=0,
        label=labels[i]
    )

plt.xlabel('Synapse volume threshold', fontsize=ML)
plt.ylabel('Median inputs per DMV neuron', fontsize=ML)
plt.xticks(volume_thresholds, [0,5,10,20,30,40,50], fontsize =LL)
plt.yticks(fontsize =LL)
plt.legend()
plt.tight_layout()
plt.savefig('DMVInputs.pdf', format='pdf')
plt.show()